# Model Evaluation

In [1]:
import torch
from torchsummary import summary
from torch.utils.data import DataLoader
from tqdm import tqdm

from evaluate import *
from loss import *
from TSDataset import *
from utils import *

## Model Summary

In [2]:
model_path = "/home/u2018144071/사기설/model/dataset_1.0_0.05_3_3_0.pt"

model = torch.load(model_path)

# select device as cuda

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device=device)

# model summary
summary(model, (3, 256, 256))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 256, 256]           1,728
       BatchNorm2d-2         [-1, 64, 256, 256]             128
              ReLU-3         [-1, 64, 256, 256]               0
            Conv2d-4         [-1, 64, 256, 256]          36,864
       BatchNorm2d-5         [-1, 64, 256, 256]             128
              ReLU-6         [-1, 64, 256, 256]               0
        DoubleConv-7         [-1, 64, 256, 256]               0
         MaxPool2d-8         [-1, 64, 128, 128]               0
            Conv2d-9        [-1, 128, 128, 128]          73,728
      BatchNorm2d-10        [-1, 128, 128, 128]             256
             ReLU-11        [-1, 128, 128, 128]               0
           Conv2d-12        [-1, 128, 128, 128]         147,456
      BatchNorm2d-13        [-1, 128, 128, 128]             256
             ReLU-14        [-1, 128, 1

## Test/validation set evaluation

In [3]:
def dataset_evaluation(model_path, data_path, data_type='test', amp=False):
    """
    Evaluate a trained model on a dataset.

    Parameters:
    - model_path (str): Path to the saved model file.
    - data_path (str): Path to the dataset.
    - data_type (str, optional): Type of data to evaluate ('test' by default).

    Returns:
    None
    """
    # Load model
    model = torch.load(model_path)
    
    # select device as cuda
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device=device)

    # Set model in evaluation mode
    model.eval()

    batch_size = 4

    # Load dataset
    test_set = TSDataset(os.path.join(data_path, data_type), transform=ToTensor())

    # Create data loader
    loader_args = dict(batch_size=batch_size, num_workers=os.cpu_count(), pin_memory=True)
    test_loader = DataLoader(test_set, shuffle=False, **loader_args)

    total_loss = 0
    amp = False

    n_channels = model.n_channels
    n_classes = model.n_classes

    criterion = BalancedMSEMAE()

    # Define evaluator for metrics
    evaluater = Evaluater(seq_len=n_classes)

    with torch.no_grad():
        # Iterate through batches for evaluation
        for batch in tqdm(test_loader, desc='Evaluating', unit='batch', leave=False):
            # Move data to device
            images, labels = batch['input'].to(device=device, dtype=torch.float32, memory_format=torch.channels_last), \
                              batch['label'].to(device=device, dtype=torch.float32)
            
            with torch.autocast(device.type if device.type != 'mps' else 'cpu', enabled=amp):
                # Forward pass
                labels_pred = model(images)
                loss = criterion(labels_pred, labels)

                # Convert to numpy arrays
                labels_pred_np = labels_pred.detach().cpu().numpy()
                labels_np = labels.cpu().numpy()

                # if save_path:
                #     save_predictions(labels_pred_np, labels_np, save_path)

                # Update evaluator with predictions and labels
                evaluater.update(labels_pred_np, labels_np)
                
                total_loss += loss.item()

        # Calculate average loss
        total_loss = total_loss / len(test_loader)

    # Print evaluation metrics
    precision, recall, f1, far, csi, hss, gss, mse, mae = evaluater.print_stat_readable()
    print(f"Test Loss: {total_loss:.4f}")

In [11]:
model_path = "/home/u2018144071/사기설/model/dataset_10.0_0.05_6_3_0.pt"
data_path = '/home/u2018144071/사기설/data/dataset_10.0_0.05_6_3_0'

# test, validation set evaluation
dataset_evaluation(model_path, data_path, data_type='valid')
dataset_evaluation(model_path, data_path, data_type='test')

INFO:root:Total Sequence Number: 306                                            
INFO:root:   TP: >0.5:1.21954e+07/1.20582e+07, >2:8.34384e+06/8.1505e+06, >5:4.49223e+06/4.1754e+06, >10:1.58719e+06/1.25094e+06, >30:128401/73134
INFO:root:   Precision: >0.5:0.660407/0.666997, >2:0.460918/0.46325, >5:0.248978/0.237678, >10:0.0842792/0.0676345, >30:0.00647906/0.00370076
INFO:root:   Recall: >0.5:0.927891/0.90824, >2:0.858068/0.826158, >5:0.764273/0.730449, >10:0.731539/0.697811, >30:0.705034/0.673469
INFO:root:   f1_score: >0.5:0.771499/0.769146, >2:0.59952/0.593633, >5:0.375581/0.358655, >10:0.150942/0.123317, >30:0.0128288/0.00736107
INFO:root:   FAR: >0.5:0.0495016/0.0590992, >2:0.0634627/0.0837298, >5:0.125117/0.184672, >10:0.303531/0.448346, >30:0.60727/0.778252
INFO:root:   CSI: >0.5:0.638374/0.640177, >2:0.446888/0.444436, >5:0.240501/0.225536, >10:0.0814769/0.0641105, >30:0.00642024/0.00365331
INFO:root:   GSS: >0.5:0.0519676/0.0693635, >2:0.0283142/0.0328728, >5:-0.00883719/-0.02

Test Loss: 2.8891


INFO:root:Total Sequence Number: 304                                            
INFO:root:   TP: >0.5:1.27707e+07/1.26423e+07, >2:9.01456e+06/8.81261e+06, >5:4.81711e+06/4.44941e+06, >10:1.50688e+06/1.15252e+06, >30:101414/49702
INFO:root:   Precision: >0.5:0.692503/0.699438, >2:0.501943/0.504496, >5:0.270813/0.257471, >10:0.0808017/0.0629301, >30:0.00514634/0.00252725
INFO:root:   Recall: >0.5:0.933847/0.915216, >2:0.865788/0.83595, >5:0.770118/0.737309, >10:0.722993/0.689043, >30:0.688997/0.664847
INFO:root:   f1_score: >0.5:0.795149/0.792909, >2:0.635295/0.629244, >5:0.400699/0.381663, >10:0.14509/0.115327, >30:0.010205/0.00503537
INFO:root:   FAR: >0.5:0.0429145/0.0508133, >2:0.0585902/0.0760522, >5:0.129695/0.191877, >10:0.334143/0.485731, >30:0.648728/0.823242
INFO:root:   CSI: >0.5:0.671585/0.674194, >2:0.486669/0.484382, >5:0.260426/0.242638, >10:0.0778677/0.0593995, >30:0.00510282/0.00249785
INFO:root:   GSS: >0.5:0.062686/0.0837761, >2:0.038617/0.0457585, >5:-0.00886076/-0.0

Test Loss: 2.9188


## Make GIF

모델을 통과시킨 결과는 다음의 과정을 거쳐서 강우 예측에 활용된다.
1. *255 : denormalization : grayscale 이미지 -> can make GIF
2. grayscale_to_dBZ
3. dBZ_to_rfrate : 강우 강도를 바탕으로 초단기예측에 활용 가능!

In [57]:
model_path = "/home/u2018144071/사기설/model/dataset_10.0_0.05_6_3_0.pt"
data_path = '/home/u2018144071/사기설/data/gif/dataset_15.0_0.05_6_6_0'

amp = False

model = torch.load(model_path)

# select device as cuda
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device=device)

# Set model in evaluation mode
model.eval()

batch_size = 1

# Load dataset
test_set = TSDataset(data_path, transform=ToTensor())

# Create data loader
loader_args = dict(batch_size=batch_size, num_workers=os.cpu_count(), pin_memory=True)
test_loader = DataLoader(test_set, shuffle=False, **loader_args)

n = 0

with torch.no_grad():
    # Iterate through batches for evaluation
    for batch in tqdm(test_loader, desc='Evaluating', unit='batch', leave=False):
        # Move data to device
        images, labels = batch['input'].to(device=device, dtype=torch.float32, memory_format=torch.channels_last), \
                            batch['label'].to(device=device, dtype=torch.float32)
        
        with torch.autocast(device.type if device.type != 'mps' else 'cpu', enabled=amp):
            # Forward pass
            labels_pred = model(images)

            # Convert to numpy arrays
            labels_pred_np = labels_pred.detach().cpu().numpy()
            labels_np = labels.cpu().numpy()
            # save_predictions. shape = (batch_size, seq_len, 256, 256)
            # loop for batch_size and seq_len
            # save with PIL Image
            # for i in range(3):
            # denormalize : multiply 255.0
            img = labels_pred_np[0, 0, :, :] * 255.0

            # chage dtype of img to uint8
            img = img.astype(np.uint8)
            img = Image.fromarray(img)
            img.save(f'/home/u2018144071/사기설/data/gif/result/{n}.png')
            
        n += 1

In [61]:
img_path = '/home/u2018144071/사기설/data/gif/truth/'
save_path = '/home/u2018144071/사기설/gt.gif'

gif = generate_gif(img_path, save_path)

display(gif)

19


In [27]:
img_path = '/home/u2018144071/사기설/data/gif/result'
save_path = '/home/u2018144071/사기설/pred.gif'

gif = generate_gif(img_path, save_path)

display(gif)

## Georeferencing

In [2]:
import rasterio

from pyproj import Proj, transform
from rasterio.transform import from_origin

In [3]:
def georeference_image(input_path, output_path, center_lon_lat, projection, pixel_size=500):
    # Example usage
    center_lon_lat = (37.5642135, 127.0016985)  # (longitude, latitude)

    pixel_size = 500
    # Load the input image
    img = Image.open(input_path)
    img = np.array(img)

    img_dBZ = grayscale_to_dBZ(img)
    img_rfrate = dBZ_to_rfrate(img_dBZ)

    # Get image dimensions
    height, width = img_rfrate.shape

    input_lon, input_lat = center_lon_lat

    # Define the source and destination coordinate systems
    wgs84 = Proj(init='epsg:4326')  # WGS84 coordinate system (latitude, longitude)
    cartesian = Proj(init=projection)  # World Mercator projection (x, y)

    # Convert latitude and longitude to Cartesian coordinates
    center_xy = transform(wgs84, cartesian, input_lat, input_lon)

    print(center_xy)

    # Calculate the top-left coordinates based on center coordinates and image size
    top_left_coords = (
        center_xy[0] - 0.5 * pixel_size * width,
        center_xy[1] + 0.5 * pixel_size * height
    )
    print(top_left_coords)

    # Create a GeoTIFF file
    with rasterio.open(
        output_path,
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,  # Set count to 1 for a single-band image
        dtype=np.float32,
        crs=projection,  # Assuming the target coordinate system is EPSG:5179
        transform=from_origin(top_left_coords[0], top_left_coords[1], pixel_size, pixel_size)
    ) as dst:
        dst.write(img_rfrate, 1)  # Assuming the image has only one band

In [7]:
# Example usage
input_path = '/home/u2018144071/사기설/15.png'
output_path = "/home/u2018144071/사기설/15epsg_3857.tif"

center_lon_lat = (37.5642135, 127.0016985)  # (longitude, latitude)

projection = 'epsg:3857'

georeference_image(input_path, output_path, center_lon_lat, projection, pixel_size=500)

(14137764.406900855, 4518045.39812495)
(14073764.406900855, 4582045.39812495)


/home/u2018144071/.conda/envs/SAGI/lib/python3.8/site-packages/pyproj/crs/crs.py:141: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)
/home/u2018144071/.conda/envs/SAGI/lib/python3.8/site-packages/pyproj/crs/crs.py:141: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)
/tmp/ipykernel_652643/2310522010.py:23: FutureWarning: This function is deprecated. See: https://pyproj4.github.io/pyproj/stable/gotchas.html#upgrading-to-pyproj-2-from-pyproj-1
  